In [1]:
from mava.networks.retention import MultiScaleRetention
from omegaconf import DictConfig
import jax
import jax.numpy as jnp
import copy

# jax.config.update("jax_enable_x64", True)

bsz = 16
num_agents = 4
obs_dim = 11
num_time_steps = 100
seq_len = num_agents * num_time_steps

retnet_embed_dim = 32
retnet_num_heads = 2
num_chunks = 1

# TODO: Recurrent form + masked test
# Is the extra added in sequence dim correct or should I do as the paper does?

2025-03-04 10:58:26.667932: W external/xla/xla/service/gpu/nvptx_compiler.cc:765] The NVIDIA driver's CUDA version is 12.4 which is older than the ptxas CUDA version (12.8.61). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
/home/ruanjohn/miniconda3/envs/mava-einops/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
memory_config = DictConfig(
    {
        "type": "rec_sable",
        "decay_scaling_factor": 0.3,
        "timestep_positional_encoding": True,
        "timestep_chunk_size": None,
    }
)

decay_kappas = 1 - jnp.exp(jnp.linspace(jnp.log(1 / 32), jnp.log(1 / 512), retnet_num_heads))
decay_kappas *= memory_config.decay_scaling_factor
decay_kappas = jnp.log(decay_kappas)
decay_kappas = decay_kappas[None, :, None, None]

In [3]:
msr = MultiScaleRetention(
    embed_dim=retnet_embed_dim,
    n_head=retnet_num_heads,
    n_agents=num_agents,
    memory_config=memory_config,
    masked=False,
    decay_scaling_factor=memory_config.decay_scaling_factor,
)

In [4]:
key = jax.random.PRNGKey(0)
key, subkey = jax.random.split(key)

obs = jax.random.normal(subkey, (bsz, seq_len, retnet_embed_dim))

# assuming no resets
dones = jnp.zeros((bsz, seq_len), dtype=bool)

init_hstate = jnp.zeros(
    (
        bsz,
        retnet_num_heads,
        retnet_embed_dim // retnet_num_heads,
        retnet_embed_dim // retnet_num_heads,
    )
)
step_counts = jnp.arange(num_time_steps)
step_counts = step_counts[None, ...].repeat(bsz, axis=0)[..., None].repeat(num_agents, axis=-1)
step_counts = step_counts.reshape(bsz, seq_len)

In [5]:
key, init_key = jax.random.split(key)
params = msr.init(
    init_key,
    obs,
    obs,
    obs,
    init_hstate,
    dones,
    step_counts,
    num_chunks=num_chunks,
)

In [6]:
hstate = copy.deepcopy(init_hstate)
act_output = []


# for the decoder we use the chunkwise
for step in range(num_time_steps):
    # todo: reset later
    hstate = hstate * jnp.exp(decay_kappas)
    obs_i = obs[:, step * num_agents : (step + 1) * num_agents, ...]
    dones_i = dones[:, step * num_agents : (step + 1) * num_agents]
    step_counts_i = step_counts[:, step * num_agents : (step + 1) * num_agents]

    out, hstate = msr.apply(
        params,
        obs_i,
        obs_i,
        obs_i,
        hstate,
        dones_i,
        step_counts_i,
        num_chunks=num_chunks,
        inference=True,
    )
    act_output.append(out)

In [7]:
act_output = jnp.concatenate(act_output, axis=1)

In [8]:
act_output.shape

(16, 400, 32)

In [9]:
hstate = copy.deepcopy(init_hstate)
train_out, _ = msr.apply(
    params, obs, obs, obs, hstate, dones, step_counts, num_chunks=1, inference=False
)

In [10]:
train_out.shape

(16, 400, 32)

In [11]:
total_error = jnp.mean(jnp.abs(train_out - act_output))
total_error

Array(6.350464e-06, dtype=float32)

In [12]:
jnp.abs(train_out - act_output)

Array([[[1.94534659e-05, 3.43471766e-06, 4.01586294e-06, ...,
         2.97650695e-06, 9.66340303e-06, 1.09262764e-05],
        [1.71735883e-05, 1.73002481e-05, 2.79396772e-07, ...,
         6.75767660e-06, 2.63992697e-05, 4.28175554e-06],
        [1.20922923e-05, 1.66986138e-05, 1.21193007e-05, ...,
         9.70810652e-06, 1.53370202e-05, 6.43171370e-06],
        ...,
        [5.46686351e-06, 5.79282641e-07, 2.36090273e-06, ...,
         3.48687172e-06, 2.86195427e-06, 4.31202352e-06],
        [2.52947211e-06, 9.52184200e-06, 1.30729750e-05, ...,
         4.77023423e-06, 6.12810254e-07, 1.83563679e-06],
        [4.11085784e-06, 5.18094748e-06, 4.24310565e-06, ...,
         6.43543899e-07, 7.90227205e-06, 2.87778676e-07]],

       [[2.16532499e-06, 1.02892518e-05, 1.23772770e-06, ...,
         2.89082527e-06, 4.15975228e-06, 6.71111047e-06],
        [2.28732824e-06, 2.36555934e-06, 7.69644976e-06, ...,
         1.99303031e-06, 1.25151128e-05, 3.32854688e-06],
        [6.84522092e-07, 